# EXAONE 동화 생성 + 선택지 프롬프트 추론 전용 Colab

이 노트북은 **학습/QLoRA/데이터셋 변환을 전부 제거한 생성 전용 버전**이다.

핵심 구조는 다음과 같다.

- EXAONE Instruct 모델을 바로 로드한다.
- 동화 장면을 생성한다.
- 선택지는 따로 학습하지 않고, 현재 장면/최종 목표/중간 목표를 넣은 프롬프트로 생성한다.
- 선택지 역할을 `행동/조사/감정`처럼 고정하지 않는다.
- 앱 연동이 쉽도록 기본 출력은 JSON으로 받는다.
- JSON이 깨지면 자동 재시도하고, 그래도 실패하면 원문을 보존한다.

기본 모델은 품질을 위해 `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct`로 설정했다.  
T4에서 VRAM이 부족하면 설정 셀에서 `LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct`로 바꾸면 된다.


In [ ]:
# 설치: 학습 관련 peft/trl/datasets는 설치하지 않는다.
!pip -q install -U "transformers>=4.43.1" "accelerate>=0.34.0" "bitsandbytes>=0.43.3" sentencepiece einops


In [ ]:
# GPU / 환경 확인
import os, platform, json, re, textwrap, random
from typing import Dict, Any, List, Optional, Union

import torch

print("python:", platform.python_version())
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("vram GB:", round(props.total_memory / 1024**3, 2))
    print("cuda capability:", props.major, props.minor)
else:
    print("GPU가 없으면 매우 느리다. Colab 런타임 > 런타임 유형 변경 > GPU를 켜라.")


## 1. 설정

- `MODEL_NAME`: EXAONE 모델 선택
- `USE_4BIT`: Colab GPU에서 VRAM을 줄이기 위해 4bit 로드
- `CHOICE_COUNT`: 앱에서 항상 3개 선택지가 필요하면 `3`, 모델이 알아서 개수를 정하게 하려면 `None`
- `MAX_HISTORY_CHARS`: 대화가 길어질 때 프롬프트에 넣을 과거 이야기 길이 제한


In [ ]:
# =========================
# 설정
# =========================

# 품질 우선 기본값. T4에서 부족하면 2.4B로 바꿔라.
MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"
# MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"

USE_4BIT = True
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 900

# 앱용이면 3 추천. 완전 자유 선택지면 None.
CHOICE_COUNT = 3
# CHOICE_COUNT = None

TEMPERATURE = 0.78
TOP_P = 0.90
REPETITION_PENALTY = 1.08

MAX_HISTORY_CHARS = 4500
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)


## 2. 모델 로드

EXAONE은 chat template을 지원하므로 `tokenizer.apply_chat_template()`을 사용한다.  
학습 코드는 없고, 여기서는 모델을 불러와 바로 생성만 한다.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


def get_compute_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major = torch.cuda.get_device_properties(0).major
    # A100/L4/H100 등은 bf16 가능. T4는 fp16이 안전하다.
    return torch.bfloat16 if major >= 8 else torch.float16

compute_dtype = get_compute_dtype()
print("compute dtype:", compute_dtype)

# Hugging Face 토큰이 필요한 경우: Colab Secrets에 HF_TOKEN 저장하거나 환경변수로 넣어라.
hf_token = os.environ.get("HF_TOKEN", None)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN") or hf_token
except Exception:
    pass

load_kwargs = dict(
    trust_remote_code=True,
    device_map="auto",
)

if torch.cuda.is_available():
    load_kwargs["torch_dtype"] = compute_dtype
else:
    load_kwargs["torch_dtype"] = torch.float32

if USE_4BIT and torch.cuda.is_available():
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=compute_dtype,
    )

if hf_token:
    load_kwargs["token"] = hf_token

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **load_kwargs)
model.eval()

print("loaded:", MODEL_NAME)
print("pad_token_id:", tokenizer.pad_token_id, "eos_token_id:", tokenizer.eos_token_id)


## 3. 공통 생성 함수

`generate_text()`는 EXAONE chat template을 적용해서 텍스트를 생성한다.  
`generate_json()`은 JSON 출력이 깨질 때 자동으로 재시도한다.


In [ ]:
# =========================
# 공통 유틸
# =========================

SYSTEM_PROMPT = """
너는 어린이용 인터랙티브 동화 앱의 전문 작가다.
출력은 자연스러운 한국어로 한다.
아이에게 과도하게 무섭거나 잔인한 장면은 쓰지 않는다.
선택지는 정해진 역할표를 따르지 말고, 현재 장면에서 실제로 자연스러운 다음 행동 후보로 만든다.
동화의 최종 목표와 현재 중간 목표에서 벗어나지 않는다.
""".strip()


def normalize_text(x: Any) -> str:
    x = str(x or "")
    x = x.replace("\r\n", "\n")
    x = re.sub(r"[ \t]+", " ", x)
    x = re.sub(r"\n{3,}", "\n\n", x)
    return x.strip()


def trim_history(text: str, max_chars: int = MAX_HISTORY_CHARS) -> str:
    text = normalize_text(text)
    if len(text) <= max_chars:
        return text
    return "...\n" + text[-max_chars:]


def apply_chat(messages: List[Dict[str, str]]) -> torch.Tensor:
    """EXAONE chat template 적용. tokenizer 버전 차이에 대응한다."""
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
        )
    except TypeError:
        # 일부 tokenizer/transformers 조합 fallback
        text = ""
        for m in messages:
            role = m.get("role", "user")
            content = m.get("content", "")
            text += f"[{role}]\n{content}\n\n"
        text += "[assistant]\n"
        return tokenizer(text, return_tensors="pt").input_ids


@torch.inference_mode()
def generate_text(
    user_prompt: str,
    system_prompt: str = SYSTEM_PROMPT,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    repetition_penalty: float = REPETITION_PENALTY,
) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = apply_chat(messages)

    # 너무 긴 입력 방지
    if input_ids.shape[-1] > MAX_INPUT_TOKENS:
        input_ids = input_ids[:, -MAX_INPUT_TOKENS:]

    input_ids = input_ids.to(model.device)
    attention_mask = torch.ones_like(input_ids)

    out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    gen_ids = out[0, input_ids.shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    """모델 출력에서 JSON object만 최대한 안전하게 추출한다."""
    text = normalize_text(text)

    # ```json ... ``` 제거
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S)
    if fenced:
        text = fenced.group(1).strip()

    # 가장 바깥 { ... } 후보 추출
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end+1]

    try:
        return json.loads(text)
    except Exception:
        return None


def validate_story_packet(obj: Dict[str, Any], choice_count: Optional[int] = CHOICE_COUNT) -> bool:
    if not isinstance(obj, dict):
        return False
    if not isinstance(obj.get("scene"), str) or len(obj.get("scene", "").strip()) < 40:
        return False
    if "choices" not in obj or not isinstance(obj["choices"], list):
        return False
    if choice_count is not None and len(obj["choices"]) != choice_count:
        return False
    if len(obj["choices"]) < 2:
        return False
    for c in obj["choices"]:
        if not isinstance(c, dict):
            return False
        if "id" not in c or "text" not in c:
            return False
        if len(str(c.get("text", "")).strip()) < 5:
            return False
    return True


def fallback_packet(raw_text: str) -> Dict[str, Any]:
    """JSON 파싱 실패 시 원문을 보존하고 최소 구조로 감싼다."""
    return {
        "scene": raw_text.strip(),
        "choices": [],
        "state": {},
        "image_prompt": "",
        "raw_text": raw_text,
        "parse_warning": "JSON 파싱에 실패해서 원문을 scene에 보존했다."
    }


@torch.inference_mode()
def generate_json(
    user_prompt: str,
    choice_count: Optional[int] = CHOICE_COUNT,
    max_retry: int = 2,
    **gen_kwargs,
) -> Dict[str, Any]:
    prompt = user_prompt
    raw = ""

    for attempt in range(max_retry + 1):
        raw = generate_text(prompt, **gen_kwargs)
        obj = extract_json_object(raw)
        if obj is not None and validate_story_packet(obj, choice_count=choice_count):
            obj["raw_text"] = raw
            obj["parse_ok"] = True
            return obj

        prompt = user_prompt + f"""

중요:
방금 출력은 앱에서 파싱하기 어렵다.
반드시 설명 없이 아래 JSON object 하나만 다시 출력해라.
문자열 안의 줄바꿈은 \\n으로 처리해도 된다.
choices는 {choice_count if choice_count is not None else '2~4'}개로 맞춰라.
""".strip()

    packet = fallback_packet(raw)
    packet["parse_ok"] = False
    return packet


## 4. 프롬프트 빌더

선택지는 여기서 프롬프트로만 만든다.  
`choice_count=3`이면 앱용 3개 선택지를 강제하되, **각 선택지의 역할/방향은 고정하지 않는다.**


In [ ]:
# =========================
# 동화/선택지 프롬프트
# =========================

JSON_SCHEMA_TEXT = """
{
  "scene": "이번 턴에 새로 생성된 동화 장면. 6~12문장 정도.",
  "choices": [
    {
      "id": 1,
      "text": "아이에게 보여줄 선택지 문장",
      "intent": "이 선택지가 이야기에서 가지는 자유로운 의미. 행동/조사/감정 같은 고정 분류 금지.",
      "expected_effect": "이 선택지를 고르면 다음 장면이 어떻게 움직일지 한 문장"
    }
  ],
  "state": {
    "current_goal": "다음 장면의 중간 목표",
    "important_clue": "현재 중요한 단서나 물건",
    "emotion": "주인공 감정",
    "progress": 0.0
  },
  "image_prompt": "이번 장면을 그릴 때 사용할 한국어 이미지 프롬프트. 등장인물/배경/분위기 중심.",
  "tts_hint": "TTS 읽기 분위기. 예: 따뜻하고 호기심 있게"
}
""".strip()


def choice_rule_text(choice_count: Optional[int]) -> str:
    if choice_count is None:
        return "선택지는 장면상 필요하면 2~4개를 만든다. 개수와 방향은 현재 장면에 맞게 정한다."
    return f"선택지는 반드시 {choice_count}개를 만든다. 단, 1번/2번/3번의 역할을 미리 고정하지 말고 각 선택지가 현재 장면에서 자연스럽게 달라지게 만든다."


def build_start_prompt(
    title: str,
    protagonist: str,
    world: str,
    final_goal: str,
    mood: str = "따뜻하고 모험적인 분위기",
    age: str = "초등학교 저학년",
    choice_count: Optional[int] = CHOICE_COUNT,
    extra_rules: str = "",
) -> str:
    return f"""
너는 인터랙티브 동화 앱의 첫 장면을 생성한다.

제목:
{title}

주인공:
{protagonist}

세계관/배경:
{world}

최종 목표:
{final_goal}

대상 독자:
{age}

분위기:
{mood}

작성 규칙:
- 첫 장면에서 주인공, 문제의 시작, 중요한 단서 하나를 자연스럽게 보여 준다.
- 최종 목표를 노골적으로 설명하지 말고 장면 속 사건으로 드러낸다.
- 너무 많은 새 인물을 한 번에 넣지 않는다.
- 아이가 다음 행동을 고르고 싶어지도록 장면 끝을 열어 둔다.
- {choice_rule_text(choice_count)}
- 선택지 문장은 짧고 명확하게 쓴다.
- 선택지의 방향은 프롬프트가 강제한 틀이 아니라 현재 장면의 흐름에서 스스로 정한다.
{extra_rules}

출력 형식:
설명, 마크다운, 코드블록 없이 JSON object 하나만 출력한다.
스키마는 아래 형태를 따른다.
{JSON_SCHEMA_TEXT}
""".strip()


def build_continue_prompt(
    title: str,
    final_goal: str,
    story_so_far: str,
    chosen_choice: Union[int, str, Dict[str, Any]],
    state: Optional[Dict[str, Any]] = None,
    mood: str = "따뜻하고 모험적인 분위기",
    age: str = "초등학교 저학년",
    choice_count: Optional[int] = CHOICE_COUNT,
    extra_rules: str = "",
) -> str:
    state = state or {}
    story_so_far = trim_history(story_so_far)
    chosen_choice_text = chosen_choice
    if isinstance(chosen_choice, dict):
        chosen_choice_text = f"{chosen_choice.get('id')}. {chosen_choice.get('text')} / 예상효과: {chosen_choice.get('expected_effect', '')}"

    return f"""
너는 인터랙티브 동화 앱의 다음 장면을 생성한다.

제목:
{title}

최종 목표:
{final_goal}

현재 상태:
{json.dumps(state, ensure_ascii=False, indent=2)}

현재까지의 이야기:
{story_so_far}

사용자가 고른 선택지:
{chosen_choice_text}

대상 독자:
{age}

분위기:
{mood}

작성 규칙:
- 사용자가 고른 선택지를 다음 장면의 원인으로 반드시 반영한다.
- 최종 목표에서 벗어나지 않게 이야기를 한 걸음 진전시킨다.
- 같은 사건을 반복하지 않는다.
- 갈등은 조금씩 해결되거나 더 선명해져야 한다.
- {choice_rule_text(choice_count)}
- 선택지의 역할표를 만들지 말고, 현재 장면에서 자연스러운 서로 다른 가능성으로 만든다.
{extra_rules}

출력 형식:
설명, 마크다운, 코드블록 없이 JSON object 하나만 출력한다.
스키마는 아래 형태를 따른다.
{JSON_SCHEMA_TEXT}
""".strip()


## 5. 세션 함수

앱에서 쓰기 쉽게 `start_story()`와 `continue_story()`로 감쌌다.


In [ ]:
# =========================
# 세션 함수
# =========================

def merge_story(story_so_far: str, new_scene: str) -> str:
    story_so_far = normalize_text(story_so_far)
    new_scene = normalize_text(new_scene)
    if not story_so_far:
        return new_scene
    return story_so_far + "\n\n" + new_scene


def find_choice(packet: Dict[str, Any], choice_id_or_text: Union[int, str]) -> Union[Dict[str, Any], str]:
    choices = packet.get("choices", []) or []
    if isinstance(choice_id_or_text, int):
        for c in choices:
            try:
                if int(c.get("id")) == choice_id_or_text:
                    return c
            except Exception:
                pass
    return str(choice_id_or_text)


def start_story(
    title: str,
    protagonist: str,
    world: str,
    final_goal: str,
    mood: str = "따뜻하고 모험적인 분위기",
    age: str = "초등학교 저학년",
    choice_count: Optional[int] = CHOICE_COUNT,
    extra_rules: str = "",
) -> Dict[str, Any]:
    prompt = build_start_prompt(
        title=title,
        protagonist=protagonist,
        world=world,
        final_goal=final_goal,
        mood=mood,
        age=age,
        choice_count=choice_count,
        extra_rules=extra_rules,
    )
    packet = generate_json(prompt, choice_count=choice_count)
    session = {
        "title": title,
        "final_goal": final_goal,
        "mood": mood,
        "age": age,
        "turn": 1,
        "story_so_far": packet.get("scene", ""),
        "state": packet.get("state", {}) or {},
        "last": packet,
        "log": [{"turn": 1, "chosen": None, "packet": packet}],
    }
    return session


def continue_story(
    session: Dict[str, Any],
    choice_id_or_text: Union[int, str],
    choice_count: Optional[int] = CHOICE_COUNT,
    extra_rules: str = "",
) -> Dict[str, Any]:
    chosen = find_choice(session.get("last", {}), choice_id_or_text)
    prompt = build_continue_prompt(
        title=session["title"],
        final_goal=session["final_goal"],
        story_so_far=session.get("story_so_far", ""),
        chosen_choice=chosen,
        state=session.get("state", {}),
        mood=session.get("mood", "따뜻하고 모험적인 분위기"),
        age=session.get("age", "초등학교 저학년"),
        choice_count=choice_count,
        extra_rules=extra_rules,
    )
    packet = generate_json(prompt, choice_count=choice_count)

    session = dict(session)
    session["turn"] = int(session.get("turn", 1)) + 1
    session["story_so_far"] = merge_story(session.get("story_so_far", ""), packet.get("scene", ""))
    session["state"] = packet.get("state", {}) or session.get("state", {})
    session["last"] = packet
    session.setdefault("log", []).append({"turn": session["turn"], "chosen": chosen, "packet": packet})
    return session


def print_packet(packet: Dict[str, Any]):
    print("\n[장면]\n")
    print(packet.get("scene", ""))
    print("\n[선택지]\n")
    for c in packet.get("choices", []) or []:
        print(f"{c.get('id')}. {c.get('text')}")
    if packet.get("image_prompt"):
        print("\n[이미지 프롬프트]\n", packet.get("image_prompt"))
    if packet.get("parse_ok") is False:
        print("\n[경고] JSON 파싱 실패. raw_text를 확인해라.")


## 6. 바로 테스트

아래 셀만 실행하면 첫 장면과 선택지가 생성된다.


In [ ]:
session = start_story(
    title="별빛 씨앗을 찾는 토끼 미루",
    protagonist="호기심 많지만 조금 겁이 많은 토끼 미루",
    world="밤이 되자 별빛 씨앗을 잃어버려 어두워진 숲. 숲속 친구들은 빛을 되찾기 위해 단서를 찾고 있다.",
    final_goal="토끼 미루가 잃어버린 별빛 씨앗을 찾아 어두워진 숲을 다시 환하게 만들고, 친구들과 힘을 합치는 법을 배운다.",
    mood="따뜻하고 신비로운 모험 분위기",
    age="초등학교 저학년",
    choice_count=CHOICE_COUNT,
)

print_packet(session["last"])


## 7. 사용자가 선택지를 골랐을 때 이어쓰기

`continue_story(session, 1)`처럼 선택지 번호를 넣으면 된다.  
앱에서 직접 선택지 텍스트를 넘겨도 된다.


In [ ]:
# 예시: 사용자가 1번 선택지를 골랐다고 가정
session = continue_story(session, 1, choice_count=CHOICE_COUNT)
print_packet(session["last"])


## 8. 간단한 콘솔 루프

Colab에서 직접 테스트하고 싶을 때 사용한다.  
중지하려면 `q`를 입력한다.


In [ ]:
# 직접 대화형 테스트용. 필요할 때만 실행.
while True:
    print_packet(session["last"])
    x = input("고를 선택지 번호 또는 직접 입력(q=종료): ").strip()
    if x.lower() in {"q", "quit", "exit"}:
        break
    if x.isdigit():
        x = int(x)
    session = continue_story(session, x, choice_count=CHOICE_COUNT)


## 9. 앱 연동용 최소 함수

서버/API에서 필요한 값만 반환하고 싶으면 아래 함수를 사용하면 된다.


In [ ]:
def app_start(payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    payload 예시:
    {
      "title": "별빛 씨앗을 찾는 토끼 미루",
      "protagonist": "토끼 미루",
      "world": "어두워진 숲",
      "final_goal": "별빛 씨앗을 찾아 숲을 밝힌다",
      "mood": "따뜻한 모험",
      "age": "초등학교 저학년",
      "choice_count": 3
    }
    """
    s = start_story(
        title=payload.get("title", "제목 없는 동화"),
        protagonist=payload.get("protagonist", "주인공"),
        world=payload.get("world", "신비로운 세계"),
        final_goal=payload.get("final_goal", "주인공이 문제를 해결하고 작은 깨달음을 얻는다."),
        mood=payload.get("mood", "따뜻한 분위기"),
        age=payload.get("age", "초등학교 저학년"),
        choice_count=payload.get("choice_count", CHOICE_COUNT),
        extra_rules=payload.get("extra_rules", ""),
    )
    return s


def app_continue(session: Dict[str, Any], selected: Union[int, str], choice_count: Optional[int] = CHOICE_COUNT) -> Dict[str, Any]:
    return continue_story(session, selected, choice_count=choice_count)


# 반환 구조 확인
# session.keys()
# session["last"]


## 운영 팁

- 선택지 품질은 학습보다 **프롬프트 + JSON 파싱 + 재시도**가 더 중요하다.
- 앱에서는 보통 `CHOICE_COUNT=3`을 유지하는 게 UI에 좋다.
- 다만 선택지의 방향은 고정하지 말아야 한다. `1번=행동`, `2번=조사`, `3번=감정` 같은 규칙을 주면 모델이 반복 패턴에 갇힌다.
- 이야기 산으로 가는 문제는 매 턴 `final_goal`, `story_so_far`, `state`, `chosen_choice`를 같이 넣어서 줄인다.
- 장면이 너무 짧으면 `MAX_NEW_TOKENS`를 올리고, 반복이 심하면 `REPETITION_PENALTY`를 1.10~1.15로 올려라.
- EXAONE 7.8B가 느리거나 VRAM이 부족하면 2.4B로 먼저 앱 플로우를 검증해라.
